# PyTorch 综合小项目

本模块把数据处理、模型、训练、评估、CUDA 与保存加载串起来。建议先完成 01～08，再独立完成这里的 TODO。

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)

# 项目一：多项式回归

目标是从带噪声数据中学习 $y=0.5x^3-2x^2+x+3$。

## 里程碑 1：特征工程与划分 ⭐⭐

生成 240 个 x，并构造 `[x, x², x³]` 三个特征。随机划分 80% 训练集和 20% 验证集，划分种子为 42。

In [ ]:
x = torch.linspace(-2, 2, 240).unsqueeze(1)
y = 0.5 * x**3 - 2 * x**2 + x + 3 + 0.1 * torch.randn_like(x)

# TODO
polynomial_features = None
full_dataset = None
train_dataset, val_dataset = None, None

assert polynomial_features.shape == (240, 3)
assert len(train_dataset) == 192 and len(val_dataset) == 48
print("✅ 项目一·里程碑 1 通过")

## 里程碑 2：训练与验证 ⭐⭐⭐

创建 DataLoader、`Linear(3,1)` 模型、MSE 损失和 Adam 优化器。训练 250 个 epoch，记录训练损失；再计算验证集 RMSE。

In [ ]:
# TODO
train_loader = None
val_loader = None
regression_model = None
loss_fn = None
optimizer = None
regression_history = []

# TODO：训练并记录每个 epoch 的平均损失

# TODO：评估验证集，计算 validation_rmse
validation_rmse = None

assert len(regression_history) == 250
assert regression_history[-1] < regression_history[0]
assert validation_rmse < 0.25
print(f"✅ 项目一完成，验证 RMSE={validation_rmse:.4f}")

# 项目二：非线性 XOR 分类

四团二维数据按对角线分成两类，线性模型无法直接完成，需要 MLP 学习非线性边界。

## 里程碑 1：生成数据 ⭐⭐

围绕四个中心各生成 100 个样本，标准差为 0.35。左下与右上标签为 0，左上与右下标签为 1。

In [ ]:
centers = torch.tensor([[-1.5, -1.5], [-1.5, 1.5], [1.5, -1.5], [1.5, 1.5]])
center_labels = torch.tensor([0, 1, 1, 0])

# TODO
feature_parts = None
label_parts = None
classification_features = None
classification_labels = None

assert classification_features.shape == (400, 2)
assert classification_labels.shape == (400,)
assert torch.equal(torch.bincount(classification_labels), torch.tensor([200, 200]))
print("✅ 项目二·里程碑 1 通过")

## 里程碑 2：训练分类器 ⭐⭐⭐

创建 `2 → 16 → 16 → 2` 的 MLP，隐藏层使用 ReLU。用 Adam 和 CrossEntropyLoss 训练 200 个 epoch，并记录损失。

In [ ]:
# TODO
classifier = None
classifier_optimizer = None
classifier_loss_fn = None
classification_history = []

features_on_device = classification_features.to(device)
labels_on_device = classification_labels.to(device)
# TODO：训练模型

classifier.eval()
with torch.no_grad():
    predictions = classifier(features_on_device).argmax(dim=1)
    classification_accuracy = (predictions == labels_on_device).float().mean().item()
assert len(classification_history) == 200
assert classification_history[-1] < classification_history[0]
assert classification_accuracy > 0.97
print(f"✅ 项目二完成，准确率={classification_accuracy:.2%}")

## 里程碑 3：混淆矩阵 ⭐⭐⭐

在当前设备上构造 2×2 混淆矩阵，行表示真实标签，列表示预测标签，然后移回 CPU。

In [ ]:
# TODO
confusion_matrix = None

assert confusion_matrix.device.type == "cpu"
assert confusion_matrix.shape == (2, 2)
assert confusion_matrix.sum().item() == 400
assert confusion_matrix.diag().sum().item() / 400 > 0.97
print("✅ 项目二·里程碑 3 通过")

# 项目三：保存、恢复与推理

保存项目二的模型，并验证重新加载后预测完全一致。

## 里程碑 1：保存模型与元数据 ⭐⭐

保存模型参数、输入维数、类别数和训练准确率。

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "xor_classifier.pt"

# TODO
artifact = None
# 保存 artifact

assert model_path.exists()
assert set(artifact) == {"model_state_dict", "input_features", "num_classes", "training_accuracy"}
print("✅ 项目三·里程碑 1 通过")

## 里程碑 2：加载并编写预测函数 ⭐⭐⭐

重新创建相同结构的模型并加载参数。补全 `predict`：接收 CPU Tensor，自动移到模型设备，返回 CPU 上的类别和概率。

In [ ]:
restored_classifier = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 2)
).to(device)

# TODO：使用 map_location=device 加载 artifact，并恢复参数


def predict(model, inputs):
    # TODO
    pass


test_points = torch.tensor([[-1.5, -1.5], [-1.5, 1.5], [1.5, -1.5], [1.5, 1.5]])
classes, probabilities = predict(restored_classifier, test_points)
assert classes.device.type == "cpu" and probabilities.device.type == "cpu"
assert torch.equal(classes, torch.tensor([0, 1, 1, 0]))
assert torch.allclose(probabilities.sum(dim=1), torch.ones(4), atol=1e-6)
assert not probabilities.requires_grad
print("✅ 项目三完成：模型可以保存、恢复并用于推理")

# 最终复盘

完成后，请用自己的话回答：

1. 为什么数据集划分必须可复现？
2. 为什么分类损失直接接收 logits？
3. 为什么验证和推理需要 `eval()` 与 `no_grad()`？
4. 为什么保存 state_dict 时，加载端还必须知道模型结构？
5. 如何让训练代码同时支持 CPU 与 CUDA？